# Phase 4 — Sensitivity Analysis & Validation
**SafeSpeed · A Digital Safe System Panel by Ventax AI Lab**

This notebook reproduces all §8 validation results and provides visual exploration.
Run `python notebooks/run_sensitivity.py` for a quick text summary without Jupyter.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
import numpy as np

with open(ROOT / 'core' / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

from core.validation import (
    load_segments_df, run_sensitivity_analysis,
    sign_detection_stats, irap_consistency_check
)

GEOJSON = ROOT / 'docs' / 'scored_segments.geojson'
df = load_segments_df(GEOJSON)
print(f'Loaded {len(df)} segments')
df[['segment_id','road_name','diagnosis','score','s_safe','posted_speed','confidence']].head(10)

## §8.3 Sensitivity Analysis

In [ ]:
sens = run_sensitivity_analysis(df, cfg)

rows = []
for name, data in sens.items():
    rows.append({
        'Variant': name,
        'Spearman rho': data['rho'],
        'Min rho': data.get('rho_min', data['rho']),
        'Max rho': data.get('rho_max', data['rho']),
        'Description': data['description']
    })

sens_df = pd.DataFrame(rows)
print('Sensitivity Analysis Results')
print('Threshold for ranking stability: rho >= 0.85')
print()
print(sens_df.to_string(index=False))
print()
print(f'Minimum rho observed: {sens_df["Min rho"].min():.4f}')
print(f'All variants >= 0.85: {(sens_df["Spearman rho"] >= 0.85).all()}')

In [ ]:
# Visualize if matplotlib available
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, ax = plt.subplots(figsize=(9, 4))
    variants = sens_df['Variant'].tolist()
    rhos = sens_df['Spearman rho'].tolist()
    mins = sens_df['Min rho'].tolist()
    maxs = sens_df['Max rho'].tolist()

    bars = ax.barh(variants, rhos, color='#3b82f6', alpha=0.8, height=0.5)
    for i, (mn, mx, r) in enumerate(zip(mins, maxs, rhos)):
        if mn != r:
            ax.plot([mn, mx], [i, i], 'o-', color='#ef4444', lw=2, ms=5)

    ax.axvline(0.85, color='#22c55e', linestyle='--', lw=1.5, label='Stability threshold (0.85)')
    ax.axvline(1.0,  color='#94a3b8', linestyle=':',  lw=1)
    ax.set_xlim(0.7, 1.05)
    ax.set_xlabel('Spearman rank correlation (rho)')
    ax.set_title('Ranking Stability Under Parameter Perturbation (S8.3)')
    ax.legend()
    for bar, rho in zip(bars, rhos):
        ax.text(rho + 0.003, bar.get_y() + bar.get_height()/2,
                f'{rho:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(ROOT / 'docs' / 'sensitivity_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to docs/sensitivity_chart.png')
except ImportError:
    print('matplotlib not installed -- text output only (see table above)')

## §8.4 Sign-Detection Cross-Check

In [ ]:
sign = sign_detection_stats(df)
for k, v in sign.items():
    if k != 'note':
        print(f'{k:<40}: {v}')
print()
print('Note:', sign['note'])

## §8.1 Internal Consistency Check (iRAP Methodology Demo)

> ⚠️ **Not independent validation.** Proxy iRAP stars are derived from the same segment features (footpath, school, PTW share) that the panel score consumes. This section demonstrates the comparison methodology only. True convergent validity requires official iRAP field ratings.

In [ ]:
irap = irap_consistency_check(df)
for k, v in irap.items():
    if k != 'note':
        print(f'{k:<40}: {v}')
print()
print('Note:', irap['note'])

## Diagnosis Distribution

In [ ]:
diag_counts = df['diagnosis'].value_counts()
print('Diagnosis distribution:')
print(diag_counts.to_string())
print()
print('Confidence distribution:')
print(df['confidence'].value_counts().to_string())
print()
print('Score statistics:')
print(df['score'].describe().round(2).to_string())

In [ ]:
try:
    import matplotlib.pyplot as plt

    colors = {
        'unsafe_limit': '#ef4444',
        'non_credible_limit': '#3b82f6',
        'design_enabled_risk': '#f97316',
        'safe': '#22c55e',
        'insufficient_data': '#94a3b8',
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    # Diagnosis pie
    vals = diag_counts.values
    lbls = diag_counts.index.tolist()
    clrs = [colors.get(l, '#94a3b8') for l in lbls]
    ax1.pie(vals, labels=[l.replace('_', ' ') for l in lbls],
            colors=clrs, autopct='%1.0f%%', startangle=140)
    ax1.set_title('Diagnosis Distribution')

    # Score histogram
    df['score'].dropna().astype(float).hist(
        ax=ax2, bins=15, color='#3b82f6', edgecolor='#0f1117', alpha=0.8
    )
    ax2.axvline(70, color='#ef4444', linestyle='--', label='High priority (>=70)')
    ax2.set_xlabel('Speed Safety Score')
    ax2.set_ylabel('Segments')
    ax2.set_title('Score Distribution')
    ax2.legend()

    plt.suptitle('Peshawar Sample — Pipeline Results', fontweight='bold')
    plt.tight_layout()
    plt.savefig(ROOT / 'docs' / 'distribution_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to docs/distribution_chart.png')
except ImportError:
    print('matplotlib not installed -- skipping charts')

## Save Results

In [ ]:
results = {'sensitivity': sens, 'sign_detection': sign, 'irap_consistency': irap}
out = ROOT / 'docs' / 'sensitivity_results.json'
with open(out, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {out}')
print('Validation report: docs/validation_report.md')